# Bayesian Changepoint Detection with multivariate data in Python

This code computes the probability of changepoints (including changes in correlation) in a time series. In this notebook I show how you can use it. This example is modified from Xiang Xuan's thesis Section 3.2.

First let's generate some data and load some modules:

In [ ]:
import matplotlib.pyplot as plt

import bayesian_changepoint_detection.generate_data as gd

%matplotlib inline
%load_ext autoreload
%autoreload 2

partition, data = gd.generate_xuan_motivating_example(200, 500)

Let's plot this data:

In [ ]:
import numpy as np

changes = partition.cumsum(0)

fig, ax = plt.subplots(figsize=[16, 4])
for p in changes:
    ax.plot([p, p], [float(data.min()), float(data.max())], "r")
for d in range(2):
    ax.plot(data[:, d])

Let's try to detect the changes with independent features:

In [ ]:
from functools import partial

from bayesian_changepoint_detection.bayesian_models import offline_changepoint_detection
from bayesian_changepoint_detection.offline_likelihoods import (
    IndependentFeaturesLikelihood,
)
from bayesian_changepoint_detection.priors import const_prior

Q_ifm, P_ifm, Pcp_ifm = offline_changepoint_detection(
    data,
    partial(const_prior, p=1 / (len(data) + 1)),
    IndependentFeaturesLikelihood(),
)

In [ ]:
fig, ax = plt.subplots(2, figsize=[18, 8])
for p in changes:
    ax[0].plot([p, p], [float(data.min()), float(data.max())], "r")
for d in range(2):
    ax[0].plot(data[:, d])
plt.legend(["Raw data with Original Changepoints"])
ax[1].plot(Pcp_ifm.exp().sum(0))
plt.legend(["Independent Factor Model"])
plt.show()

Unfortunately, not very good... Now let's try the full covariance model (warning, it'll take a while):

In [ ]:
from bayesian_changepoint_detection.offline_likelihoods import FullCovarianceLikelihood

Q_full, P_full, Pcp_full = offline_changepoint_detection(
    data,
    partial(const_prior, p=1 / (len(data) + 1)),
    FullCovarianceLikelihood(),
)

In [ ]:
fig, ax = plt.subplots(2, figsize=[18, 8])
for p in changes:
    ax[0].plot([p, p], [float(data.min()), float(data.max())], "r")
for d in range(2):
    ax[0].plot(data[:, d])
plt.legend(["Raw data with Original Changepoints"])
ax[1].plot(Pcp_full.exp().sum(0))
plt.legend(["Full Covariance Model"])
plt.show()

Ahh, much better now!

In [ ]:
%timeit Q_ifm, P_ifm, Pcp_ifm = offline_changepoint_detection(data, partial(const_prior, p=1/(len(data)+1)), IndependentFeaturesLikelihood())